#Log Probability Zero-shot MGT Detector

##GPT2-XL Backbone

In [ ]:
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import os, math, random, numpy as np, pandas as pd, torch
import json
import ast


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2-xl"
MAX_LEN = 512

print(f"Source model: {MODEL_NAME} on {DEVICE}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# BEEMO dataset

print("\n Loading BEEMO dataset...")
beemo = load_dataset("toloka/beemo")
df = beemo["train"].to_pandas()
print(f" Loaded {len(df)} samples")


Source model: gpt2-xl on cuda

 Loading BEEMO dataset...
 Loaded 2187 samples


In [ ]:
# Mean log-prob

def mean_logprob(text: str) -> float:
    try:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        ).to(DEVICE)

        if enc.input_ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            out = model(**enc).logits

        logits = out[:, :-1, :]
        labels = enc.input_ids[:, 1:]
        attn = enc.attention_mask[:, 1:]

        log_probs = torch.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
        mean_lp = (tok_lp * attn).sum(dim=1) / (attn.sum(dim=1) + 1e-8)
        return float(mean_lp.detach().cpu().item())
    except Exception as e:
        return np.nan


def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    X.append(str(val).strip())
                    y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y


In [ ]:
def evaluate_logprob(texts, labels):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = mean_logprob(t)
        scores.append(s if np.isfinite(s) else -1e9)

    thr = float(np.median([s for s in scores if s > -1e8])) if any(s > -1e8 for s in scores) else 0.0
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)

    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return scores, preds, {"accuracy": acc, "auroc": auroc, "precision": prec, "recall": rec, "f1": f1}

TASKS = {
    "H_vs_M": ("human_output", "model_output"),
    "E_vs_M": ("human_edits", "model_output"),
    "H_vs_E": ("human_output", "human_edits"),
    "L_vs_M": ("llama-3.1-70b_edits", "model_output"),
    "H_vs_L": ("human_output", "llama-3.1-70b_edits"),
    "G_vs_M": ("gpt-4o_edits", "model_output"),
    "H_vs_G": ("human_output", "gpt-4o_edits"),
}

PROJECT_DIR = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

results = {}
for name, (c1, c2) in TASKS.items():
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    texts, labels = prepare(df, c1, c2)
    print(f"Samples: {len(texts)} ({labels.count(0)}/{labels.count(1)})")
    scores, preds, metrics = evaluate_logprob(texts, labels)
    results[name] = metrics
    pd.DataFrame({"text": texts, "label": labels, "score": scores, "pred": preds}).to_csv(
        f"{PROJECT_DIR}/results/{name}_logprob_gpt2xl.csv", index=False
    )
    print(f"AUROC: {metrics['auroc']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

summary = pd.DataFrame(results).T
summary.to_csv(f"{PROJECT_DIR}/results/logprob_gpt2xl_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (GPT-2 XL)")
print("="*60)
print(summary[["auroc", "accuracy", "f1"]].round(4))
print(f"\n Results saved to {PROJECT_DIR}/results/")



H_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [04:02<00:00, 18.07it/s]


AUROC: 0.6965 | Acc: 0.6408 | F1: 0.6408

E_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [04:11<00:00, 17.36it/s]


AUROC: 0.6421 | Acc: 0.6015 | F1: 0.6014

H_vs_E
Samples: 4374 (2187/2187)


...

AUROC: 0.5898 | Acc: 0.5482 | F1: 0.6386

G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [06:29<00:00, 22.48it/s]


AUROC: 0.6017 | Acc: 0.5488 | F1: 0.3984

H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [06:07<00:00, 23.82it/s]


AUROC: 0.6131 | Acc: 0.5610 | F1: 0.6488

FINAL SUMMARY (GPT-2 XL)
         auroc  accuracy      f1
H_vs_M  0.6965    0.6408  0.6408
E_vs_M  0.6421    0.6015  0.6014
H_vs_E  0.5709    0.5544  0.5543
L_vs_M  0.6356    0.5638  0.4183
H_vs_L  0.5898    0.5482  0.6386
G_vs_M  0.6017    0.5488  0.3984
H_vs_G  0.6131    0.5610  0.6488

Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##OPT-1.3B Backbone

In [1]:
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import os, math, random, numpy as np, pandas as pd, torch
import json
import ast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
MAX_LEN = 512

print(f"Source model: {MODEL_NAME} on {DEVICE}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,

    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# BEEMO dataset

print("\n Loading BEEMO dataset...")
beemo = load_dataset("toloka/beemo")
df = beemo["train"].to_pandas()
print(f" Loaded {len(df)} samples")


Source model: facebook/opt-1.3b on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

...


 Loading BEEMO dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2187 [00:00<?, ? examples/s]

 Loaded 2187 samples


In [2]:
# Mean log-prob

def mean_logprob(text: str) -> float:
    try:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        ).to(DEVICE)

        if enc.input_ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            out = model(**enc).logits

        logits = out[:, :-1, :]
        labels = enc.input_ids[:, 1:]
        attn = enc.attention_mask[:, 1:]

        log_probs = torch.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
        mean_lp = (tok_lp * attn).sum(dim=1) / (attn.sum(dim=1) + 1e-8)
        return float(mean_lp.detach().cpu().item())
    except Exception as e:
        return np.nan


def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    X.append(str(val).strip())
                    y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y


In [3]:
def evaluate_logprob(texts, labels):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = mean_logprob(t)
        scores.append(s if np.isfinite(s) else -1e9)

    thr = float(np.median([s for s in scores if s > -1e8])) if any(s > -1e8 for s in scores) else 0.0
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)

    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return scores, preds, {"accuracy": acc, "auroc": auroc, "precision": prec, "recall": rec, "f1": f1}

TASKS = {
    "H_vs_M": ("human_output", "model_output"),
    "E_vs_M": ("human_edits", "model_output"),
    "H_vs_E": ("human_output", "human_edits"),
    "L_vs_M": ("llama-3.1-70b_edits", "model_output"),
    "H_vs_L": ("human_output", "llama-3.1-70b_edits"),
    "G_vs_M": ("gpt-4o_edits", "model_output"),
    "H_vs_G": ("human_output", "gpt-4o_edits"),
}

PROJECT_DIR = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

results = {}
for name, (c1, c2) in TASKS.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    texts, labels = prepare(df, c1, c2)
    print(f"Samples: {len(texts)} ({labels.count(0)}/{labels.count(1)})")
    scores, preds, metrics = evaluate_logprob(texts, labels)
    results[name] = metrics
    pd.DataFrame({"text": texts, "label": labels, "score": scores, "pred": preds}).to_csv(
        f"{PROJECT_DIR}/results/{name}_logprob_opt1.3b.csv", index=False
    )
    print(f"AUROC: {metrics['auroc']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

summary = pd.DataFrame(results).T
summary.to_csv(f"{PROJECT_DIR}/results/logprob_opt1.3b_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (OPT1.3B)")
print("="*60)
print(summary[["auroc", "accuracy", "f1"]].round(4))
print(f"\n Results saved to {PROJECT_DIR}/results/")



 H_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [02:37<00:00, 27.75it/s]


AUROC: 0.7175 | Acc: 0.6616 | F1: 0.6616

 E_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [02:50<00:00, 25.66it/s]


AUROC: 0.6615 | Acc: 0.6155 | F1: 0.6155

 H_vs_E
Samples: 4374 (2187/2187)


...

AUROC: 0.5862 | Acc: 0.5424 | F1: 0.6339

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [04:26<00:00, 32.88it/s]


AUROC: 0.6093 | Acc: 0.5521 | F1: 0.4027

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [04:08<00:00, 35.25it/s]


AUROC: 0.6327 | Acc: 0.5730 | F1: 0.6584

 FINAL SUMMARY (OPT1.3B)
         auroc  accuracy      f1
H_vs_M  0.7175    0.6616  0.6616
E_vs_M  0.6615    0.6155  0.6155
H_vs_E  0.5723    0.5505  0.5505
L_vs_M  0.6640    0.5813  0.4417
H_vs_L  0.5862    0.5424  0.6339
G_vs_M  0.6093    0.5521  0.4027
H_vs_G  0.6327    0.5730  0.6584

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##Falcon-7B Backbone

In [ ]:
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import os, math, random, numpy as np, pandas as pd, torch
import json
import ast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "tiiuae/falcon-7b"
MAX_LEN = 512
print(f"Source model: {MODEL_NAME} on {DEVICE}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True,)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# BEEMO dataset

print("\n Loading BEEMO dataset...")
beemo = load_dataset("toloka/beemo")
df = beemo["train"].to_pandas()
print(f" Loaded {len(df)} samples")


Source model: tiiuae/falcon-7b on cuda


config.json: 0.00B [00:00, ?B/s]

configuration_falcon.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b:
- configuration_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.



modeling_falcon.py: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]


 Loading BEEMO dataset...
 Loaded 2187 samples


In [ ]:
# Mean log-prob

def mean_logprob(text: str) -> float:
    try:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        ).to(DEVICE)

        if enc.input_ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            out = model(**enc).logits

        logits = out[:, :-1, :]
        labels = enc.input_ids[:, 1:]
        attn = enc.attention_mask[:, 1:]

        log_probs = torch.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
        mean_lp = (tok_lp * attn).sum(dim=1) / (attn.sum(dim=1) + 1e-8)
        return float(mean_lp.detach().cpu().item())
    except Exception as e:
        return np.nan

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    X.append(str(val).strip())
                    y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y


In [ ]:
def evaluate_logprob(texts, labels):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = mean_logprob(t)
        scores.append(s if np.isfinite(s) else -1e9)

    thr = float(np.median([s for s in scores if s > -1e8])) if any(s > -1e8 for s in scores) else 0.0
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)

    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return scores, preds, {"accuracy": acc, "auroc": auroc, "precision": prec, "recall": rec, "f1": f1}

TASKS = {
    "H_vs_M": ("human_output", "model_output"),
    "E_vs_M": ("human_edits", "model_output"),
    "H_vs_E": ("human_output", "human_edits"),
    "L_vs_M": ("llama-3.1-70b_edits", "model_output"),
    "H_vs_L": ("human_output", "llama-3.1-70b_edits"),
    "G_vs_M": ("gpt-4o_edits", "model_output"),
    "H_vs_G": ("human_output", "gpt-4o_edits"),
}

PROJECT_DIR = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

results = {}
for name, (c1, c2) in TASKS.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    texts, labels = prepare(df, c1, c2)
    print(f"Samples: {len(texts)} ({labels.count(0)}/{labels.count(1)})")
    scores, preds, metrics = evaluate_logprob(texts, labels)
    results[name] = metrics
    pd.DataFrame({"text": texts, "label": labels, "score": scores, "pred": preds}).to_csv(
        f"{PROJECT_DIR}/results/{name}_logprob_falcon7b.csv", index=False
    )
    print(f"AUROC: {metrics['auroc']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

summary = pd.DataFrame(results).T
summary.to_csv(f"{PROJECT_DIR}/results/logprob_falcon7b_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY")
print("="*60)
print(summary[["auroc", "accuracy", "f1"]].round(4))
print(f"\n Results saved to {PROJECT_DIR}/results/")



 H_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [03:39<00:00, 19.89it/s]


AUROC: 0.6952 | Acc: 0.6401 | F1: 0.6401

 E_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [03:44<00:00, 19.49it/s]


AUROC: 0.6506 | Acc: 0.6088 | F1: 0.6087

 H_vs_E
Samples: 4374 (2187/2187)


...

AUROC: 0.5704 | Acc: 0.5351 | F1: 0.6281

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [06:35<00:00, 22.12it/s]


AUROC: 0.5948 | Acc: 0.5472 | F1: 0.3963

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [06:24<00:00, 22.73it/s]


AUROC: 0.6187 | Acc: 0.5634 | F1: 0.6508

 FINAL SUMMARY
         auroc  accuracy      f1
H_vs_M  0.6952    0.6401  0.6401
E_vs_M  0.6506    0.6088  0.6087
H_vs_E  0.5606    0.5407  0.5406
L_vs_M  0.6520    0.5765  0.4351
H_vs_L  0.5704    0.5351  0.6281
G_vs_M  0.5948    0.5472  0.3963
H_vs_G  0.6187    0.5634  0.6508

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
